# Lesson 04 - Tool Use Design Pattern

In this lesson you will learn the **Tool Use** design pattern for AI agents using the Microsoft Agent Framework (Python). We cover:

- Defining function tools with the `@tool` decorator and typed parameters
- Providing tool schemas so the model knows what each tool does
- Controlling tool execution with `approval_mode`
- Returning **structured output** via Pydantic models and `response_format`

The scenario is a **travel booking agent** that can look up destinations, check availability, and retrieve flight information.

## Setup

> **My copy:** runs on **MiniMax** via `get_chat_client()` (see `my-work/common/llm.py`) instead of Microsoft Foundry + `az login`.
> Requires `MINIMAX_API_KEY` in the repo-root `.env`.

In [1]:
# (deps are installed in the repo-root .venv from requirements.txt)

In [2]:
import sys
sys.path.insert(0, "../")  # lets this notebook import my-work/common/llm.py

import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from common.llm import get_chat_client, parse_json_response, strip_think

C:\Users\Balaji Calluru\Documents\Learning\AI\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\Balaji Calluru\Documents\Learning\AI\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


In [3]:
# Swapped from the course original: FoundryChatClient + DefaultAzureCredential -> helper
client = get_chat_client()

Using provider: minimax


## Defining Tools with the @tool Decorator

The `@tool` decorator turns a plain Python function into a tool that an agent can call.
Key points:

- The **docstring** becomes the tool description the model sees.
- **Type annotations** (including `Annotated` with descriptions) define the tool schema.
- `approval_mode` controls whether the user must approve each call before it executes.

In [4]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creating an Agent with Multiple Tools

Pass all three tools to the client so the model can invoke whichever ones it needs to answer the user's question.

In [5]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

I'll check the available destinations and their booking status.<think>Now check availability for each destination.</think>

Here's what I have:

**All destinations (6):** Barcelona, Paris, Berlin, Tokyo, Sydney, New York City

**Availability status:**

| Destination | Status |
|---|---|
| 🇪🇸 Barcelona | ✅ Available — *3 spots left* |
| 🇫🇷 Paris | ✅ Available |
| 🇩🇪 Berlin | ❌ Sold out |
| 🇯🇵 Tokyo | ✅ Available — *1 spot left* |
| 🇦🇺 Sydney | ✅ Available |
| 🗽 New York City | ✅ Available |

**5 of 6 destinations are bookable.** Berlin is currently sold out, so I can't confirm availability for it right now.

A couple of things worth flagging:
- **Tokyo is down to just 1 spot** — if you're interested, that's the one most likely to sell out next.
- **Paris and Sydney show plain "Available"** without a spot count, so I don't have visibility into how many remain — you may want to act sooner on those.

Would you like me to pull flight information for any of these? Just let me know your origi

## Structured Output with Tools

The course original passes a Pydantic model as `response_format`. **MiniMax does not honor `response_format`** - it replies in prose and Pydantic raises a `ValidationError` (I hit this on the first run).

**Workaround used here:** tell the agent to reply with JSON only, matching the schema, then parse the reply with `parse_json_response()` from `common/llm.py`. It also strips MiniMax's inline `<think>` blocks. This is the pattern the capstone will use for structured decisions.

In [6]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. "
        "Reply with a single JSON object ONLY (no prose, no markdown fences) matching this schema: "
        '{"recommendations": [{"destination": str, "available": bool, '
        '"flight_details": str, "estimated_cost": int}]}'
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
plan = parse_json_response(str(response), TravelPlan)
for rec in plan.recommendations:
    status = "available" if rec.available else "unavailable"
    print(f"{rec.destination}: {status} - {rec.flight_details} (~${rec.estimated_cost})")

Barcelona: available - BA 2042, Departs 08:30, Arrives 11:45; 3 spots left (~$350)
Paris: available - AF 1081, Departs 09:15, Arrives 11:30 (~$280)
Berlin: unavailable - No direct flights from LHR to BER (~$0)


## Tool Approval Patterns

The `approval_mode` parameter on `@tool` controls whether tool calls require human approval before executing:

| Mode | Behaviour |
|---|---|
| `"never_require"` | Tool runs automatically — no user confirmation needed. |
| `"always_require"` | Every call must be approved by the user before it executes. |

Use `"always_require"` for tools that have side-effects (e.g. booking a flight, charging a credit card) so a human stays in the loop.

In [7]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

Tool name: book_flight
Approval mode: always_require


## Summary

In this lesson you learned how to:

1. **Define tools** using the `@tool` decorator with typed parameters and docstrings that serve as the tool schema.
2. **Compose multiple tools** so the agent can call them in sequence to answer complex queries.
3. **Return structured output** by passing a Pydantic model as `response_format`.
4. **Control tool approval** with `approval_mode` to keep a human in the loop for sensitive operations.

These patterns form the foundation for building reliable, production-ready agents that can interact with external systems safely.